# Cycle Benchmarking (Cycle B)

Estimate the process fidelity of a **dressed cycle** `G̃R̃` and the per-cycle error
`r = 1 − F_RC`, SPAM-robustly — resolving the direction-dependent Pauli fidelities `λ_P`
that Standard RB averages away.

Spec & cell plan: see [`CB.md`](CB.md).

In [26]:
# ── Cell 1 — Imports ──────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import itertools
from dataclasses import dataclass
from typing import Any
import time

# Quantum info: Paulis, Cliffords, channel representations
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import (
    Pauli, PauliList, Clifford, Operator,
    SuperOp, PTM, DensityMatrix, random_pauli,
)

# Aer: simulator, noise channels, sampler primitive (hardware-portable path)
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel, QuantumError, ReadoutError,
    depolarizing_error, pauli_error, coherent_unitary_error,
)
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeFez
from qiskit_ibm_runtime import SamplerV2 as RuntimeSampler
rng = np.random.default_rng(42)

In [27]:
# Cell 2 - Helpers
# make_cycle
def make_cycle(gate_name, n_qubits):
    # return the QC with the gate appended
    qc = QuantumCircuit(n_qubits)
    if gate_name == "I":
        order = 1
    elif gate_name == "s":
        order = 4
        for qbit in range(n_qubits):
            qc.s(qbit)
    elif gate_name == "cz":
        order = 2
        for qbit in range(0,  n_qubits-1, 2):
            qc.cz(qbit, qbit + 1)
    elif gate_name == "H":
        for qbit in range(n_qubits):
            qc.h(qbit)
        order = 2
    
    return qc, order

def make_noise(noise, n_qubits, label):
    kind = noise["kind"]
    full_error = None
    if kind == "correlated":
        full_error = pauli_error(list(noise["probs"].items()))
    else:
        if kind == "depolarizing":
            error = depolarizing_error(noise["p"], 1)
        elif kind == "pauli":
            error = pauli_error(list(noise["probs"].items()))
        full_error = error
        
        for _ in range(n_qubits -1):
            full_error = full_error.tensor(error)
    
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(full_error, label)
    return nm, full_error.to_quantumchannel()  

# create the pauli comnibations based on qubit size
# for 1 qubit = 4 paulis and 2 qubit =15 paulis use all of them as K
# for 3 qubits onwards trim the list to get a smaller K then len(paulis)

def make_paulis(n_qubits, rng, K=None):

    paulis = [''.join(p) for p in itertools.product("IXYZ", repeat = n_qubits)]
    paulis.remove("I"*n_qubits) # remove the identity
    if n_qubits>2 and K is not None and K<len(paulis):
        return [paulis[i] for i in rng.choice(len(paulis), size=K, replace=False)]
    else:
        return paulis # return as is do not trim for 1 and 2 qubits
    

In [28]:
# calculate expected lambdas for different noise models
'''noises = {
    "depolarizing": {"kind":"depolarizing", "p":0.05},
    "dephasing":    {"kind":"pauli", "probs":{"I":0.95, "Z":0.05}},
    "bit_flip":     {"kind":"pauli", "probs":{"I":0.95, "X":0.05}},
    "biased":       {"kind":"pauli", "probs":{"I":0.90, "X":0.03, "Y":0.02, "Z":0.05}},
}'''


@dataclass
class CBConfig:
     n_qubits: int
     shots: int
     m1: int
     m2: int
     L: int
     K: int
     label: str
     rng: np.random.Generator
     factorization: bool = False
     transpile_fn: Any = None

# helpers for expected lambdas here

def to_probs(noise):
    if noise["kind"] == "depolarizing": 
        # see above depol doesnt have probs
        p = noise["p"]
        return {"I": 1-3*p/4, "X": p/4, "Y": p/4, "Z": p/4}
    return noise["probs"]

def expected_lambda(probs, P):
    P = Pauli(P)
    return sum(prob * (1 if P.commutes(Pauli(Q)) else -1) for Q, prob in probs.items())

# predicts what is lambda_P if noise were independent
def expected_lambda_orbit(probs, P, G_map):
    # ---- Step 1: FIND THE ORBIT (this loop is the orbit-finding) ----
    orbit, Q = [], Pauli(P)
    while True:
        label = Q.to_label().lstrip("+-i") # drop the sign
        orbit.append(label) # append to orbit
        Q = Q.evolve(G_map, frame="s") # sandiwch G with Q
        if Q.to_label().lstrip("+-i") == P: # string sign again and check if reached a circle, if yes stop. we found the orbit.
            break
    # ---- Step 2: BARE λ at each spot (per-qubit product) ----
    bare_lambdas = [np.prod([expected_lambda(probs, op) for op in label]) for label in orbit]

    # ---- Step 3: GEOMEAN (root = orbit size) ----
    return np.prod(bare_lambdas) ** (1 / len(orbit))    

In [ ]:
# build the ideal cycle C(P) and the real_noisy_cycle C̃(ρ) for a given Pauli P, sequence length m and random sequence index l

def calculate_ideal_circuit(P, qc):
    output = Pauli(P).evolve(qc, frame='s')
    assert output.phase in (0,2)
    return 1 if output.phase == 0 else -1
    

def prep_eigenstate(qc, P):
    for qbit, op in enumerate(P[::-1]): # we pass pauli as XZ but qiskit thinks Z is first qubit ie q0 so we reverse this by [::-1]
        if op == "X":
            qc.h(qbit)
        elif op == "Y":
            qc.h(qbit)
            qc.s(qbit)
    
def apply_dressed(qc, random_paulis, G):
    for i , random_pauli in enumerate(random_paulis):
        for qbit, op in enumerate(random_pauli[::-1]):
            if op == "X":
                qc.x(qbit)
            elif op == "Y":
                qc.y(qbit)
            elif op == "Z":
                qc.z(qbit)
        if i < len(random_paulis)-1:
            qc.append(G, range(qc.num_qubits))


def measure_P(qc, P):
    for qbit, op in enumerate(P[::-1]):
        if op == "X":
            qc.h(qbit)
        elif op == "Y":
            qc.sdg(qbit)
            qc.h(qbit)
    qc.measure(range(qc.num_qubits), range(qc.num_qubits))


def build_real_circuit(P, m, random_paulis, n_qubits, G_gate):
    qc = QuantumCircuit(n_qubits, n_qubits)
    prep_eigenstate(qc, P)
    apply_dressed(qc, random_paulis, G_gate)
    measure_P(qc, P)
    return qc

def run_once(P, m, cfg, G_map, G_gate, sampler):
    # this f_pml = Tr[C(P), C̃(ρ)] is the core of the protocol: one random sequence, one shot at the overlap
    random_paulis = ["".join(cfg.rng.choice(list("IXYZ"), cfg.n_qubits)) for _ in range(m+1)]

    run_qc = build_real_circuit(P, m, random_paulis, cfg.n_qubits, G_gate)

    if cfg.transpile_fn is not None:
        run_qc = cfg.transpile_fn(run_qc)

    ideal_qc = QuantumCircuit(cfg.n_qubits)
    apply_dressed(ideal_qc, random_paulis, G_map) # check this
    pauli_sign = calculate_ideal_circuit(P, ideal_qc)

    results = sampler.run([run_qc], shots=cfg.shots).result()
    counts = results[0].data.c.get_counts()
    expectation_value = get_expectation_value(counts, P, cfg.shots)
    return pauli_sign * expectation_value

def get_expectation_value(counts, P, shots):
    P = P[::-1] # reverse to match qiskit's ordering
    total = 0
    for bitstring, c in counts.items():
        bitstring = bitstring[::-1] # reverse to match qiskit's ordering
        eigenvalue = 1
        for i in range(len(P)):
            if P[i] !="I" and bitstring[i] =='1':
                eigenvalue *= -1
        total += eigenvalue * c
    return total / shots

'''P has only ±1 eigenvalues. The loop's whole job is: for each measured bitstring, decide whether it's the +1 outcome or the −1 outcome — then average them. ⟨P⟩ is the average of those ±1's, so a number in [−1, +1].

1. For 1 qubit it's trivial (P="Z"): bit "0" → +1, bit "1" → −1. With counts={"0":95,"1":5}:

<Z> = (95·(+1) + 5·(−1)) / 100 = 0.9

That's all. The loop for n=1 literally does (-1)^0 = +1 for "0" and (-1)^1 = −1 for "1".

2. For multiple qubits, a single bitstring's eigenvalue isn't one bit — it's the product of each qubit's ±1. After measure_P rotates P→Z…Z, each qubit gives +1 (bit 0) or −1 (bit 1), and P's eigenvalue is their product:

eigenvalue = (+1 or −1) × (+1 or −1) × ...   over P's support
           = (−1)^(number of 1-bits on the support)
    
'''

def run_cb(paulis_K, cfg, G_map, G_gate, sampler):
    lambdas = {}
    t0 = time.time()
    for i, P in enumerate(paulis_K):
        F = {}
        for m in (cfg.m1, cfg.m2):
            F[m] = np.mean([run_once(P, m, cfg, G_map, G_gate, sampler) for _ in range(cfg.L)])
        lambdas[P] = (F[cfg.m2]/F[cfg.m1]) ** (1/(cfg.m2-cfg.m1))
        elapsed = time.time()-t0
        print(f"[{i+1:>2}/{len(paulis_K)}] P={P:<3}  λ={lambdas[P]:.4f}  (elapsed {elapsed:.1f}s)", flush=True)
    return lambdas

def print_lambdas(name, measured, expected=None):
    print(f"\n  {name}")
    if expected is None:
        print(f"  {'P':<5}{'measured':>10}")
        for P in measured:
            print(f"  {P:<5}{measured[P]:>10.4f}")
        print(f"  {'F_RC':<5}{np.mean(list(measured.values())):>10.4f}")
    else:    
        print(f"  {'P':<5}{'expected':>10}{'measured':>10}{'Δ':>9}")
        for P in expected:
            print(f"  {P:<5}{expected[P]:>10.4f}{measured[P]:>10.4f}{measured[P]-expected[P]:>+9.4f}")
        e, m = np.mean(list(expected.values())), np.mean(list(measured.values()))
        print(f"  {'F_RC':<5}{e:>10.4f}{m:>10.4f}{m-e:>+9.4f}")

# 2 qubit correlated errors need a test method.
def factorization_test(lambdas, cfg):
    print(f"  {'P':<6}{'λ_P(measured)':>15}{'λ_P(expected)':>15}{'Δ':>9}")
    for P, lambda_P in lambdas.items():
        if P.count("I") >= cfg.n_qubits-1:
            continue # skip 1-qubit marginals
        lambda_pred_P = 1.0
        for i, op in enumerate(P):
            if op!="I":
                marginal_P = "I"*i + op + "I"*(cfg.n_qubits-1-i) 
                # XX: i=0→"XI", i=1→"IX"
                lambda_pred_P *= lambdas[marginal_P]
        print(f"  {P:<6}{lambda_P:>15.4f}{lambda_pred_P:>15.4f}{lambda_P - lambda_pred_P:>+9.4f}")


In [30]:
# the run experiment 

def run_experiment(cfg, cycle_gates, noises):
    paulis = make_paulis(cfg.n_qubits, cfg.rng, cfg.K)
    for cycle_gate in cycle_gates:
        G_map, order = make_cycle(cycle_gate, cfg.n_qubits)
        G_gate = UnitaryGate(Operator(G_map).data, label=cfg.label)
        print("==================================================================")
        print(f"Cycle {cycle_gate} (order {order}), {cfg.n_qubits} qubit — expected vs measured λ_P")
        # run the cycle benckhmarking 
        for name, noise in noises.items():
            # expected results
            if (noise["kind"] in ("depolarizing", "pauli")):
                probs = to_probs(noise)
                expected_lambdas = {P: expected_lambda_orbit(probs, P, G_map) for P in paulis}
            elif noise["kind"] == "correlated":
                probs = noise["marginal_probs"]
                expected_lambdas = {P: expected_lambda_orbit(probs, P, G_map) for P in paulis}
                
            else:
                expected_lambdas = None

            # now the real circuit run with noise
            nm, channel = make_noise(noise, cfg.n_qubits, cfg.label)
            sampler = AerSampler(options = {"backend_options":{"noise_model": nm}})
            measured_lambdas = run_cb(paulis, cfg, G_map, G_gate, sampler)
            print_lambdas(name, measured_lambdas, expected_lambdas)
            if cfg.n_qubits > 1 and cfg.factorization:
                factorization_test(measured_lambdas, cfg)


In [31]:
# Terminology used throughout the notebook:
#   ideal_cycle = C(P)    : ideal expected outcome, a signed Pauli ±P   (classical, always exact)
#   real_cycle  = C̃(ρ)    : noisy output from running the SAME circuit  (sim or hardware)
#   overlap   f_{P,m,l} = Tr[ ideal_cycle · real_cycle ]                (= +1 if noiseless)

# --- Noise on G. CB measures lambda_P = survival of P-component:  Lambda(P) = lambda_P * P.
#"Channels preserve I" is the statement λ_I = 1 — and that's automatic, regardless of p_I: λ_I = p_I + p_X + p_Y + p_Z = (1−q) + q = 1 
#
# lambda_P = P(errors COMMUTING with P) - P(errors ANTICOMMUTING with P)
# commute: X~I,X | Y~I,Y | Z~I,Z ; each anticommutes with the other two)
# lambda_I = 1 <- always; I never decays, so not measured
#       lambda_X = (pI+pX) - (pY+pZ)
#       lambda_Y = (pI+pY) - (pX+pZ)
#       lambda_Z = (pI+pZ) - (pX+pY)
# 
# expected CB output (verify against this): order is (λ_X, λ_Y, λ_Z)
# interestingly e.g Z damages X and Y and Z is left alone e.g. dephasing
#   depolarizing p {"kind":"depolarizing","p":p} -> (1-p, 1-p, 1-p)
#   dephasing-q {"kind":"pauli","probs":{"I":1-q,"Z":q}} -> (1-2q,  1-2q,  1)
#   bit-flip-q {"kind":"pauli","probs":{"I":1-q,"X":q}} -> (1,1-2q,  1-2q)
#   biased {"I":.90,"X":.03,"Y":.02,"Z":.05} -> (0.86,  0.84,  0.90)
CYCLE_LABEL = "CB_Cycle"
noises = {
    "depolarizing": {"kind":"depolarizing", "p":0.05},
    "dephasing":    {"kind":"pauli", "probs":{"I":0.95, "Z":0.05}},
    "bit_flip":     {"kind":"pauli", "probs":{"I":0.95, "X":0.05}},
    "biased":       {"kind":"pauli", "probs":{"I":0.90, "X":0.03, "Y":0.02, "Z":0.05}},
}

correlated_noise = {"correlated_ZZ": {"kind":"correlated", "probs":{"II":0.95, "ZZ":0.05}, "marginal_probs":{"I":0.95, "Z":0.05}}}

cfg = CBConfig(n_qubits=1, shots=100, m1=4, m2=12, L=30, K=30, rng=rng, label=CYCLE_LABEL)
run_experiment(cfg, ["I", "H"], noises)

Cycle I (order 1), 1 qubit — expected vs measured λ_P
[ 1/3] P=X    λ=0.9546  (elapsed 0.1s)
[ 2/3] P=Y    λ=0.9534  (elapsed 0.2s)
[ 3/3] P=Z    λ=0.9502  (elapsed 0.3s)

  depolarizing
  P      expected  measured        Δ
  X        0.9500    0.9546  +0.0046
  Y        0.9500    0.9534  +0.0034
  Z        0.9500    0.9502  +0.0002
  F_RC     0.9500    0.9527  +0.0027
[ 1/3] P=X    λ=0.9122  (elapsed 0.1s)
[ 2/3] P=Y    λ=0.9010  (elapsed 0.2s)
[ 3/3] P=Z    λ=1.0000  (elapsed 0.3s)

  dephasing
  P      expected  measured        Δ
  X        0.9000    0.9122  +0.0122
  Y        0.9000    0.9010  +0.0010
  Z        1.0000    1.0000  +0.0000
  F_RC     0.9333    0.9377  +0.0044
[ 1/3] P=X    λ=1.0000  (elapsed 0.1s)
[ 2/3] P=Y    λ=0.9029  (elapsed 0.2s)
[ 3/3] P=Z    λ=0.9074  (elapsed 0.3s)

  bit_flip
  P      expected  measured        Δ
  X        1.0000    1.0000  +0.0000
  Y        0.9000    0.9029  +0.0029
  Z        0.9000    0.9074  +0.0074
  F_RC     0.9333    0.9368  +0.0034

In [32]:
cfg = CBConfig(n_qubits=2, shots=200, m1=4, m2=8, L=30, K=30, rng=rng, label=CYCLE_LABEL)
run_experiment(cfg, ["I"], {**noises, **correlated_noise}) # local gate

Cycle I (order 1), 2 qubit — expected vs measured λ_P
[ 1/15] P=IX   λ=0.9461  (elapsed 0.1s)
[ 2/15] P=IY   λ=0.9539  (elapsed 0.2s)
[ 3/15] P=IZ   λ=0.9465  (elapsed 0.3s)
[ 4/15] P=XI   λ=0.9532  (elapsed 0.5s)
[ 5/15] P=XX   λ=0.9011  (elapsed 0.6s)
[ 6/15] P=XY   λ=0.9008  (elapsed 0.7s)
[ 7/15] P=XZ   λ=0.9047  (elapsed 0.9s)
[ 8/15] P=YI   λ=0.9517  (elapsed 1.0s)
[ 9/15] P=YX   λ=0.9079  (elapsed 1.1s)
[10/15] P=YY   λ=0.9089  (elapsed 1.2s)
[11/15] P=YZ   λ=0.9031  (elapsed 1.4s)
[12/15] P=ZI   λ=0.9469  (elapsed 1.5s)
[13/15] P=ZX   λ=0.8981  (elapsed 1.6s)
[14/15] P=ZY   λ=0.9048  (elapsed 1.7s)
[15/15] P=ZZ   λ=0.9084  (elapsed 1.9s)

  depolarizing
  P      expected  measured        Δ
  IX       0.9500    0.9461  -0.0039
  IY       0.9500    0.9539  +0.0039
  IZ       0.9500    0.9465  -0.0035
  XI       0.9500    0.9532  +0.0032
  XX       0.9025    0.9011  -0.0014
  XY       0.9025    0.9008  -0.0017
  XZ       0.9025    0.9047  +0.0022
  YI       0.9500    0.9517  +0.00

## Independent vs correlated noise — the "alarm" picture

Picture each qubit's Z-error as a little **alarm that goes off 5% of the time** (`q = 0.05`).

**Independent noise = two separate alarms.** Each qubit has its own, unrelated alarm.
- Qubit 0's alarm rings 5% of the time.
- Qubit 1's alarm rings 5% of the time.
- **Both at the same moment?** 5% × 5% = **0.25%** — almost never. You mostly see one qubit err at a time.

**Correlated ZZ noise = one alarm wired to both qubits.** A single alarm; when it rings (5%), **both qubits flip together**, otherwise nothing.
- Qubit 0 errs 5% of the time. Qubit 1 errs 5% of the time.
- **Both at the same moment? 5%** — same alarm. You *never* see just one qubit flip alone.

| | one qubit errs | both err together |
|---|---|---|
| Independent (two alarms) | 5% | **0.25%** (rare) |
| Correlated (one wired alarm) | 5% | **5%** (common) |

Watching one qubit at a time, the two look identical — each errs 5%. The difference shows only when you watch them **together**:
- both err ≈ 0.25% → independent, no crosstalk
- both err ≈ 5%, far more than chance → correlated, crosstalk

The excess of "both-together" beyond what chance (0.25%) allows **is** the correlation.


## Which Paulis break factorization, and by how much

Pls note: XZ: \(XZ = -ZX\) (or \(XZ = -iY\))ZX: \(ZX = -XZ\) (or \(ZX = iY\))ZY: \(ZY = -YZ\) (or \(ZY = -iX\))YZ: \(YZ = -ZY\) (or \(YZ = iX\))XY: \(XY = -YX\) (or \(XY = iZ\)

Setup: cycle `G = I`, noise = correlated ZZ error `{II: 1−q, ZZ: q}` (q = 0.05) attached to G. We read `λ_P` for each 2-qubit Pauli `P`.

**The one rule.** A `Z` error **flips** a measurement of `X` or `Y` on that qubit (→ −1) and is **invisible** to `Z` or `I` (→ +1). The `ZZ` error hits *both* qubits at once, so its net effect on a joint measurement `P = P₀⊗P₁` is **(qubit-0 flip) × (qubit-1 flip)**.

### Three worked examples

**① P = XX** (measure X on both)
- Qubit 0: measuring X, Z hits it → flip → **−1**
- Qubit 1: measuring X, Z hits it → flip → **−1**
- Net = (−1)(−1) = **+1** → no net flip → **λ_XX = 1.0**
- The ZZ error is *invisible* to XX — the two flips cancel.

**② P = XI** (measure X on qubit 0, nothing on qubit 1) — a marginal
- Qubit 0: measuring X → flip → **−1**
- Qubit 1: measuring I → Z is invisible → **+1**
- Net = (−1)(+1) = **−1** → flips every time → **λ_XI = 1 − 2q = 0.90**

**③ P = XZ** (measure X on qubit 0, Z on qubit 1)
- Qubit 0: measuring X → flip → **−1**
- Qubit 1: measuring Z → Z is invisible → **+1**
- Net = (−1)(+1) = **−1** → flips → **λ_XZ = 0.90**
- Check factorization: marginals λ_XI = 0.90, λ_IZ = 1.0 → product 0.90 × 1.0 = 0.90 = λ_XZ. **Holds.** ✓

**The decay factor `1 − 2q`.** `⟨P⟩` is an average of ±1 readings. For a `P` the error *net-flips*:

| | prob | reading |
|---|---|---|
| no error | 1 − q | +1 |
| error fires | q | −1 (flipped) |

```
λ_P = (1−q)·(+1) + q·(−1) = 1 − 2q
```
A flip is a *reversal* (+1 → −1, a swing of 2), so flipping a fraction `q` drops the average by `2q` — hence `1 − 2q`, not `1 − q`.  (q = 0.05 → **λ = 0.90**.)
If the error does **not** net-flip `P`, every reading stays +1 even when it fires → `λ_P = (1−q) + q = 1`.

**Factorization check** (joint vs product of single-qubit marginals):
- `λ_XX = 1.0` but `λ_XI · λ_IX = 0.90 × 0.90 = 0.81` → **Δ = +0.19, breaks** 💡
- `λ_XZ = 0.90` and `λ_XI · λ_IZ = 0.90 × 1.0 = 0.90` → Δ = 0, holds ✓

The break is the crosstalk fingerprint: the ZZ error damages `X` on each qubit *alone* (`λ_XI = λ_IX = 0.90`) yet leaves `XX` *untouched* (`λ_XX = 1.0`). Independent noise can't do this — two separate errors would make `XX` decay *more* (0.81), never less.

**Complete classification.** A site is *flippable* if it's `X`/`Y` (a `Z` kick flips it), *immune* if it's `I`/`Z`:

| both sites | Paulis | flips | λ | factorizes? |
|---|---|---|---|---|
| both immune | `ZI, IZ, ZZ` | 0 | 1.0 | ✓ holds (1 = 1·1) |
| one flippable, one immune | `XI, IX, YI, IY` (marginals); `XZ, ZX, YZ, ZY` (joints) | 1 | 0.90 | ✓ holds (0.90 = 0.90·1) |
| both flippable | `XX, XY, YX, YY` | 2 → cancel | 1.0 | ✗ **breaks** (1.0 ≠ 0.81) 💡 |

Exactly **4 Paulis light up** — the all-`X`/`Y` block `{XX, XY, YX, YY}` — and they are the signature of the ZZ correlation.


In [33]:
cfg = CBConfig(n_qubits=2, shots=200, m1=4, m2=8, L=30, K=30, rng=rng, label=CYCLE_LABEL)
run_experiment(cfg, ["cz"], {"bit_flip": noises["bit_flip"]})  # entangling cycle, anisotropic noise

Cycle cz (order 2), 2 qubit — expected vs measured λ_P
[ 1/15] P=IX   λ=0.9496  (elapsed 0.1s)
[ 2/15] P=IY   λ=0.8462  (elapsed 0.2s)
[ 3/15] P=IZ   λ=0.8966  (elapsed 0.3s)
[ 4/15] P=XI   λ=0.9531  (elapsed 2.4s)
[ 5/15] P=XX   λ=0.8957  (elapsed 2.5s)
[ 6/15] P=XY   λ=0.8939  (elapsed 2.6s)
[ 7/15] P=XZ   λ=0.9373  (elapsed 2.7s)
[ 8/15] P=YI   λ=0.8361  (elapsed 2.8s)
[ 9/15] P=YX   λ=0.8949  (elapsed 2.9s)
[10/15] P=YY   λ=0.8962  (elapsed 2.9s)
[11/15] P=YZ   λ=0.8485  (elapsed 3.0s)
[12/15] P=ZI   λ=0.8947  (elapsed 3.1s)
[13/15] P=ZX   λ=0.9442  (elapsed 3.2s)
[14/15] P=ZY   λ=0.8529  (elapsed 3.3s)
[15/15] P=ZZ   λ=0.8209  (elapsed 3.4s)

  bit_flip
  P      expected  measured        Δ
  IX       0.9487    0.9496  +0.0009
  IY       0.8538    0.8462  -0.0076
  IZ       0.9000    0.8966  -0.0034
  XI       0.9487    0.9531  +0.0045
  XX       0.9000    0.8957  -0.0043
  XY       0.9000    0.8939  -0.0061
  XZ       0.9487    0.9373  -0.0113
  YI       0.8538    0.8361  -0.0177


In [34]:
cfg = CBConfig(n_qubits=2, shots=200, m1=4, m2=8, L=30, K=30, rng=rng, label=CYCLE_LABEL)
run_experiment(cfg, ["cz"], correlated_noise)  # entangling cycle, ZZ noise

Cycle cz (order 2), 2 qubit — expected vs measured λ_P
[ 1/15] P=IX   λ=0.9092  (elapsed 0.1s)
[ 2/15] P=IY   λ=0.9075  (elapsed 0.2s)
[ 3/15] P=IZ   λ=1.0000  (elapsed 0.3s)
[ 4/15] P=XI   λ=0.9063  (elapsed 0.4s)
[ 5/15] P=XX   λ=1.0000  (elapsed 0.5s)
[ 6/15] P=XY   λ=1.0000  (elapsed 0.6s)
[ 7/15] P=XZ   λ=0.8996  (elapsed 0.6s)
[ 8/15] P=YI   λ=0.8960  (elapsed 0.7s)
[ 9/15] P=YX   λ=1.0000  (elapsed 0.8s)
[10/15] P=YY   λ=1.0000  (elapsed 0.9s)
[11/15] P=YZ   λ=0.9029  (elapsed 1.0s)
[12/15] P=ZI   λ=1.0000  (elapsed 1.1s)
[13/15] P=ZX   λ=0.9031  (elapsed 1.2s)
[14/15] P=ZY   λ=0.8950  (elapsed 1.2s)
[15/15] P=ZZ   λ=1.0000  (elapsed 1.3s)

  correlated_ZZ
  P      expected  measured        Δ
  IX       0.9000    0.9092  +0.0092
  IY       0.9000    0.9075  +0.0075
  IZ       1.0000    1.0000  +0.0000
  XI       0.9000    0.9063  +0.0063
  XX       0.8100    1.0000  +0.1900
  XY       0.8100    1.0000  +0.1900
  XZ       0.9000    0.8996  -0.0004
  YI       0.9000    0.8960  -0.

In [35]:
# N=3 bit flip and entangling cycle CZ

cfg = CBConfig(n_qubits=3, shots=200, m1=4, m2=8, L=30, K=30, rng=rng, label=CYCLE_LABEL)
run_experiment(cfg, ["cz"], {"bit_flip": noises["bit_flip"]})

Cycle cz (order 2), 3 qubit — expected vs measured λ_P
[ 1/30] P=IXX  λ=0.8942  (elapsed 0.2s)
[ 2/30] P=IXY  λ=0.9078  (elapsed 0.3s)
[ 3/30] P=YZI  λ=0.8028  (elapsed 0.5s)
[ 4/30] P=IZX  λ=0.9505  (elapsed 0.7s)
[ 5/30] P=YYX  λ=0.8098  (elapsed 0.8s)
[ 6/30] P=XXI  λ=0.9490  (elapsed 1.0s)
[ 7/30] P=IXZ  λ=0.9473  (elapsed 1.2s)
[ 8/30] P=IIZ  λ=0.9020  (elapsed 1.3s)
[ 9/30] P=YXI  λ=0.8587  (elapsed 1.4s)
[10/30] P=ZXX  λ=0.7879  (elapsed 1.6s)
[11/30] P=YYZ  λ=0.7693  (elapsed 1.7s)
[12/30] P=IYX  λ=0.9148  (elapsed 1.9s)
[13/30] P=XXZ  λ=0.9462  (elapsed 2.0s)
[14/30] P=ZYY  λ=0.7694  (elapsed 2.2s)
[15/30] P=ZIZ  λ=0.8097  (elapsed 2.4s)
[16/30] P=YII  λ=0.8961  (elapsed 2.5s)
[17/30] P=IYI  λ=0.8650  (elapsed 2.7s)
[18/30] P=IYY  λ=0.8989  (elapsed 2.9s)
[19/30] P=IIY  λ=0.8422  (elapsed 3.0s)
[20/30] P=ZZX  λ=0.8599  (elapsed 3.2s)
[21/30] P=ZYX  λ=0.7881  (elapsed 3.4s)
[22/30] P=YXX  λ=0.8275  (elapsed 3.5s)
[23/30] P=XZZ  λ=0.8409  (elapsed 3.7s)
[24/30] P=YXY  λ=0.8185  

In [36]:
# helpers for hardware runs (transpilation, factorization test) here

@dataclass
class HWConfig:
    backend: Any
    sampler: Any
    physical_qubits: list
    tag: str = ""

def pick_cz_gate_pairs(backend, max_error=0.5):
    """Filter dead pairs (err >= max_err) and return ranked working CZ edges per IBM calibration.
    Returns dict {'clean': (c), 'noisy': (n)}.
    """
    cz = backend.target["cz"]
    seen, edges = set(), []
    for qpair, props in cz.items(): 
        key = tuple(sorted(qpair))  # de-dup symmetric (a,b)/(b,a) both r same pair
        if key in seen:
            continue
        seen.add(key)
        err = getattr(props, "error", None)
        if err is None or err >=max_error:
            continue
        edges.append((list(key), err))
    edges.sort(key=lambda x: x[1]) # sort by error err
    return {"clean" : edges[0], "noisy": edges[-1]}
    #edges[0]        : ([22, 23], 2.14e-3)    ← cleanest
    #edges[-1]       : ([153, 154], 5.00e-2)  ← noisiest

def run_experiment_hw(cbcfg, cycle_gate, hwcfg):
    """Hardware variant of run_experiment.
        Differences vs sim run_experiment:
      - No noise injection — the device IS the noise channel.
      - No expected_lambdas — no probs to feed the orbit predictor.
      - factorization_test is the only detector (self-consistency, needs no truth).
      - Transpiles each circuit onto physical_qubits in backend's native basis.
    """
    paulis = make_paulis(cbcfg.n_qubits, cbcfg.rng, cbcfg.K)
    G_map, order = make_cycle(cycle_gate, cbcfg.n_qubits)
    G_gate = UnitaryGate(Operator(G_map).data, label = cbcfg.label)

    # Build pass manager
    pass_mgr = generate_preset_pass_manager(
        backend=hwcfg.backend,
        optimization_level=1, # dont collaps dressed cycle
        initial_layout=hwcfg.physical_qubits,
    )
    cbcfg.transpile_fn = pass_mgr.run

    print("==================================================================")
    print(f"Cycle {cycle_gate} on {hwcfg.backend.name}, qubits {hwcfg.physical_qubits} {hwcfg.tag}")

    try:
        measured_lambdas = run_cb(paulis, cbcfg, G_map, G_gate, hwcfg.sampler)
    finally:
        cbcfg.transpile_fn = None # reset to default for next runs


    print_lambdas(f"{hwcfg.backend.name}/{hwcfg.physical_qubits}", measured_lambdas)

    ibm_err = hwcfg.backend.target["cz"][tuple(hwcfg.physical_qubits)].error
    f_rc = np.mean(list(measured_lambdas.values()))
    print(f"  F_RC: {f_rc:.4f} vs IBM's CZ error {ibm_err:.4f}")
    if cbcfg.n_qubits > 1 and cbcfg.factorization:
        factorization_test(measured_lambdas, cbcfg)
    
    return measured_lambdas


def estimate_circuits(cfg, n_pairs=1, n_cycle_gates=1):
    """Pre-flight: print total circuits + shots for a CB hardware run."""
    if cfg.n_qubits <= 2:
        n_paulis = 4**cfg.n_qubits - 1                # 3 for N=1, 15 for N=2
    else:
        n_paulis = min(cfg.K, 4**cfg.n_qubits - 1)    # K-subset for N≥3
    n_lengths = 2                                      # m1, m2
    per_pair = n_paulis * n_lengths * cfg.L
    total_circ = per_pair * n_pairs * n_cycle_gates
    total_shot = total_circ * cfg.shots
    print(f"  Paulis (K)         : {n_paulis}")
    print(f"  Lengths            : {n_lengths}  (m1={cfg.m1}, m2={cfg.m2})")
    print(f"  Draws per (P,m)    : {cfg.L}")
    print(f"  Pairs              : {n_pairs}")
    print(f"  Cycle gates        : {n_cycle_gates}")
    print(f"  ────────────────────")
    print(f"  Circuits per pair  : {per_pair:,}")
    print(f"  TOTAL CIRCUITS     : {total_circ:,}")
    print(f"  Shots per circuit  : {cfg.shots}")
    print(f"  TOTAL SHOTS        : {total_shot:,}")
    return total_circ, total_shot



In [ ]:
fez = FakeFez()
cz_gate_pairs = pick_cz_gate_pairs(fez)
noisy_pair, noisy_err = cz_gate_pairs["noisy"]
clean_pair, clean_err = cz_gate_pairs["clean"]
print(f"Clean pair {clean_pair} with error {clean_err:.4f}")
print(f"Noisy pair {noisy_pair} with error {noisy_err:.4f}")

sampler = RuntimeSampler(mode=fez)

cbcfg = CBConfig(n_qubits=2, shots=200, m1=4, m2=8, L=5, K=30, rng=rng, label=CYCLE_LABEL, factorization=True)

estimate_circuits(cbcfg, n_pairs=2)

run_experiment_hw(cbcfg, "cz", HWConfig(backend=fez, sampler=sampler, physical_qubits=noisy_pair, tag="noisy"))
run_experiment_hw(cbcfg, "cz", HWConfig(backend=fez, sampler=sampler, physical_qubits=clean_pair, tag="clean"))


Clean pair [22, 23] with error 0.0021
Noisy pair [153, 154] with error 0.0500
Cycle cz on fake_fez, qubits [153, 154] noisy
[ 1/15] P=IX   λ=0.9896  (elapsed 95.4s)
[ 2/15] P=IY   λ=0.9249  (elapsed 191.8s)
[ 3/15] P=IZ   λ=1.0065  (elapsed 287.3s)
[ 4/15] P=XI   λ=0.9408  (elapsed 385.2s)
[ 5/15] P=XX   λ=0.9198  (elapsed 481.2s)
[ 6/15] P=XY   λ=0.9155  (elapsed 576.9s)
[ 7/15] P=XZ   λ=0.9480  (elapsed 671.0s)
[ 8/15] P=YI   λ=0.9022  (elapsed 766.9s)
[ 9/15] P=YX   λ=0.9423  (elapsed 863.6s)
[10/15] P=YY   λ=0.9272  (elapsed 960.4s)
[11/15] P=YZ   λ=0.9552  (elapsed 1057.6s)
[12/15] P=ZI   λ=0.9279  (elapsed 2207.8s)
[13/15] P=ZX   λ=0.9814  (elapsed 3392.2s)
[14/15] P=ZY   λ=0.9486  (elapsed 3932.8s)
[15/15] P=ZZ   λ=0.9683  (elapsed 4029.1s)

  fake_fez/[153, 154]
  P      measured
  IX       0.9896
  IY       0.9249
  IZ       1.0065
  XI       0.9408
  XX       0.9198
  XY       0.9155
  XZ       0.9480
  YI       0.9022
  YX       0.9423
  YY       0.9272
  YZ       0.9552
  Z

(300, 60000)